# 02 — Explore and Validate Data

> **SYNTHETIC DATA DEMO** — All telemetry and fault scenarios in this notebook are programmatically generated. Results demonstrate architecture and controlled functional behaviour, not real-world accuracy or statistical validation.

Checks schema/data quality and deterministic physical invariants before any analytics or LLM reasoning.

In [1]:
# Colab/local bootstrap: discover the repository, clone only when needed.
from pathlib import Path
import os, sys, subprocess

start = Path.cwd().resolve()
candidates = [start, *start.parents]
ROOT = next((p for p in candidates if (p / "src").exists() and (p / "config").exists()), None)
if ROOT is None:
    if Path("/content").exists():
        repo = Path("/content/intelligent-data-logger-demo")
        if not repo.exists():
            subprocess.run(["git", "clone", "https://github.com/Engr-Daniel/intelligent-data-logger-demo.git", str(repo)], check=True)
        ROOT = repo
    else:
        raise RuntimeError("Could not locate the repository. Open the notebook from the cloned repo or use Colab.")
os.chdir(ROOT)
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

Repository root: /mnt/data/m5work/intelligent-data-logger-demo


In [2]:
from src.reasoning.tools import load_demo_context
from src.datacontext.context import load_schema
from src.datacontext.validation import validate_telemetry

df, cfg = load_demo_context()
report = validate_telemetry(df, cfg, load_schema())
print("status:", report["status"])
print("power-balance coverage:", report["coverage"]["power_balance"])
print("battery-transition coverage:", report["coverage"]["battery_transition"])
print("data-quality events:", report["data_quality_events"])

status: PASS
power-balance coverage: {'eligible_rows': 34551, 'excluded_rows': 9, 'coverage_pct': 99.97395833333333}
battery-transition coverage: {'evaluated_rows': 24830, 'excluded_missing_rows': 11, 'structural_non_evaluable_rows': 9719, 'input_coverage_pct': 99.97106397754565, 'equation_applicability_pct': 71.84814375415955, 'sufficiency_basis': 'input_coverage_pct', 'pass_meaning': 'Sufficient input telemetry was available and no violations were found among transitions to which the M2 flow-only equation applies.'}
data-quality events: [{'event_type': 'telemetry_unavailable', 'rows_affected': 9, 'fields_affected': ['battery_charge_w', 'battery_discharge_w', 'battery_soc_pct', 'battery_stored_energy_wh', 'battery_temp_c', 'cloud_cover_pct', 'grid_export_w', 'grid_import_w', 'inverter_dc_current_a', 'inverter_dc_voltage_v', 'inverter_efficiency', 'inverter_temp_c', 'irradiance_wm2', 'pv_ac_power_w', 'pv_dc_power_w'], 'detected_from': 'observed_missing_values'}]


In [3]:
assert report["status"] == "PASS"
assert report["details"]["power_balance_residual_w"]["violations"] == 0
print("Physical/data-context validation passed for the controlled demo.")

Physical/data-context validation passed for the controlled demo.
